# Pixel-wise runoff-onset vs climate anomaly correlations (exploratory)

Correlates the runoff-onset anomaly against each ERA5-Land variable's anomaly per
pixel across water years, on an Equal-Earth grid **derived on the fly**
(`era5.combined_anomaly_eqearth`: ERA5 stack → EPSG:8857, coarse onset from the
public pyramid, masked + merged, anomalies vs the all-year median). The standing
eqearth stores this used to read are retired. Output: the correlations zarr at
`era5.correlations_store_path(config)` (local, under `scratch/`) — this notebook's own product,
read by nothing else.

In [ ]:
import xarray as xr
from global_snowmelt_runoff_onset.config import Config
from gsro_analysis import era5, settings
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

In [ ]:
# The correlation product, streamed one (variable, water year) slab at a time (~3 GB peak, no dask
# cluster): era5.pixelwise_correlations has the exact semantics of combined_anomaly_eqearth + xr.corr
# along water_year, which as one dask graph over the ~20 GB stack never converged on a 16 GB box
# (2026-09-03 memory pass). ~20 GB of ERA5 reads: tens of minutes.
corr_path = era5.pixelwise_correlations(config)
corr_path

In [ ]:
# the correlations zarr is now at era5.correlations_store_path(config) (local, under scratch/)
print(era5.correlations_store_path(config))

In [ ]:
# derived on the fly — the standing eqearth stores are retired (2026-08-24)
combined_runoff_and_era5_10yr_anomaly_ds = era5.combined_anomaly_eqearth(config)
combined_runoff_and_era5_10yr_anomaly_ds

In [ ]:
combined_runoff_and_era5_10yr_anomaly_ds['temperature_2m'].sel(month='spring_month_1').compute().plot.imshow(col='water_year', col_wrap=5,vmin=-10,vmax=10,cmap='RdBu_r')

In [ ]:
runoff_onset_and_era5_10yr_anomaly_correlations_ds = xr.open_zarr(
    era5.correlations_store_path(config),   # local, under scratch/
    decode_coords='all', chunks='auto')
runoff_onset_and_era5_10yr_anomaly_correlations_ds

In [ ]:
runoff_onset_and_era5_10yr_anomaly_correlations_ds = runoff_onset_and_era5_10yr_anomaly_correlations_ds.compute()
runoff_onset_and_era5_10yr_anomaly_correlations_ds

In [ ]:
runoff_onset_and_era5_10yr_anomaly_correlations_ds['temperature_2m'].plot.imshow(col='month',col_wrap=3,cmap='RdBu')

In [ ]:
#WUS_bbox = minx=-130, miny=30, maxx=-60, maxy=75
WUS_bbox = [-130, 30, -60, 75]
HMA_bbox = [65, 25, 110, 45]
northern_europe_and_asia_bbox = [-10, 45, 80, 75]
bbox = HMA_bbox
region_correlations_ds = runoff_onset_and_era5_10yr_anomaly_correlations_ds.rio.clip_box(*bbox, crs="EPSG:4326")
region_correlations_ds

In [ ]:
# don't need surface_net_solar_radiation_sum or surface_net_thermal_radiation_sum
# with RdBu colormap, red means increase in var means earlier runoff onset, blue means increase in var means later runoff onset

In [ ]:
import easysnowdata
import pandas as pd

In [ ]:
easysnowdata.utils.datetime_to_DOWY(pd.Timestamp('2024-03-01'))

In [ ]:
easysnowdata.utils.datetime_to_DOWY(pd.Timestamp('2024-06-01'))

In [ ]:
easysnowdata.utils.datetime_to_DOWY(pd.Timestamp('2024-07-01'))

In [ ]:
import cartopy.crs as ccrs


In [ ]:


for var in region_correlations_ds.data_vars:
    print(var)
    fig = region_correlations_ds[var].plot.imshow(col='month',col_wrap=3,cmap='RdBu',sharex=True,sharey=True,subplot_kws={'projection': ccrs.EqualEarth()},figsize=(12,8))
    for ax in fig.axs.flatten():
        ax.set_aspect('equal')
        ax.set_title(ax.get_title().replace('month = ',''))
        ax.gridlines(draw_labels=False)
        #ctx.add_basemap(ax,crs=ccrs.EqualEarth())
    fig.fig.suptitle(f'Correlation between 10-year runoff onset anomaly and\n10-year anomaly in ERA5-Land {var}', y=1.00)
#     plt.show()
# fig = region_correlations_ds[var].plot.imshow(col='month',col_wrap=3,cmap='RdBu',sharex=True,sharey=True,subplot_kws={'projection': ccrs.EqualEarth()},figsize=(12,8))
# for ax in fig.axs.flatten():
#     ax.set_aspect('equal')
#     ax.set_title(ax.get_title().replace('month = ',''))
#     ax.gridlines(draw_labels=False)
#     #ctx.add_basemap(ax,crs=ccrs.EqualEarth())
# fig.fig.suptitle(f'Correlation between 10-year runoff onset anomaly and\n10-year anomaly in ERA5-Land {var}', y=1.00)

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)
gmba_gdf

In [ ]:
mountain_range_name = "Sierra Nevada"
mountain_range_name = "Olympic Mountains"
mountain_range_name = "Brooks Range"

In [ ]:
mountain_range_gdf = gmba_gdf[gmba_gdf['MapName']==mountain_range_name]
mountain_range_gdf

In [ ]:
mountain_range_correlations_ds = runoff_onset_and_era5_10yr_anomaly_correlations_ds.rio.clip(mountain_range_gdf.to_crs(runoff_onset_and_era5_10yr_anomaly_correlations_ds.rio.crs).geometry)
mountain_range_correlations_ds

In [ ]:
mountain_range_correlations_ds['surface_solar_radiation_downwards_sum'].plot.imshow(col='month',col_wrap=6)

In [ ]:

var = 'temperature_2m'
fig = mountain_range_correlations_ds[var].plot.imshow(col='month',col_wrap=6,vmin=-1,vmax=1,cmap='RdBu_r',figsize=(20,5))
for i, ax in enumerate(fig.axs.flatten()[:mountain_range_correlations_ds.sizes['month']]):   # col_wrap=6 leaves 3 empty axes after 9 months
    mountain_range_gdf.to_crs(mountain_range_correlations_ds.rio.crs).boundary.plot(ax=ax, color='black', linewidth=1)
    avg_correlation = mountain_range_correlations_ds[var].mean(dim=['x','y'])
    month = ax.get_title().split("=")[-1]
    ax.set_title(f'{month}\navg_corr={avg_correlation.values[i]:.2f}')
    ax.set_aspect('equal')
    ax.axis('off')

In [ ]:
for var in mountain_range_correlations_ds.data_vars:
    fig = mountain_range_correlations_ds[var].plot.imshow(col='month',col_wrap=6,vmin=-1,vmax=1,cmap='RdBu_r',figsize=(12,5))
    for i, ax in enumerate(fig.axs.flatten()[:mountain_range_correlations_ds.sizes['month']]):   # col_wrap=6 leaves 3 empty axes after 9 months
        mountain_range_gdf.to_crs(mountain_range_correlations_ds.rio.crs).boundary.plot(ax=ax, color='black', linewidth=1)
        avg_correlation = mountain_range_correlations_ds[var].mean(dim=['x','y'])
        month = ax.get_title().split("=")[-1]
        ax.set_title(f'{month}\navg_corr={avg_correlation.values[i]:.2f}')
        ax.axis('off')
    fig.fig.suptitle(var,y=1.02)

In [ ]:
# derived on the fly — the standing eqearth stores are retired (2026-08-24)
combined_runoff_and_era5_10yr_anomaly_ds = era5.combined_anomaly_eqearth(config)
combined_runoff_and_era5_10yr_anomaly_ds

In [ ]:
mountain_range_mean_anomalies = combined_runoff_and_era5_10yr_anomaly_ds.rio.clip(mountain_range_gdf.to_crs(combined_runoff_and_era5_10yr_anomaly_ds.rio.crs).geometry).mean(['x','y'])
mountain_range_mean_anomalies

In [ ]:
mountain_range_mean_anomalies.sel(month='spring_month_1').plot.scatter(x='temperature_2m', y='runoff_onset')

In [ ]:
mountain_range_mean_anomalies['runoff_onset'].plot()
mountain_range_mean_anomalies['temperature_2m'].sel(month='spring_month_1')

In [ ]:
for month in mountain_range_mean_anomalies['month'].values:
    corr = xr.corr(mountain_range_mean_anomalies['temperature_2m'].sel(month=month), mountain_range_mean_anomalies['runoff_onset']).compute().values
    print(f'Correlation for {month}: {corr}')
    

In [ ]:
corr = xr.corr(mountain_range_mean_anomalies['temperature_2m'].sel(month=['spring_month_1', 'spring_month_2','spring_month_3']).mean(dim='month'), mountain_range_mean_anomalies['runoff_onset']).compute().values
print(f'Correlation for spring months: {corr}')
